# From Classifications to Decisions: Donor Targeting with Machine Learning

## Overview

CharityAI is a mid-sized nonprofit that must decide **whom to contact** and **how to allocate limited outreach resources**, all under operational constraints. The organization has demographic and historical data on potential donors and wants to use machine learning to raise more donations.

This project walks you through that process in three stages: first you build predictive models using a proxy label (Stage 1) and a behavioral label (Stage 2), then you turn those predictions into spatially-aware outreach decisions under routing and budget constraints (Stage 3). A central lesson throughout:

> **A good prediction model does not automatically lead to a good decision.**

**What to submit:** This notebook, completed, with all code cells run and all written answers filled in. Verify with Kernel → Restart & Run All before submitting.

---
## Setup

Run the cell below to import libraries and set global parameters. **Do not modify this cell.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, roc_curve, classification_report
)
from sklearn.model_selection import train_test_split
import folium
from geopy.distance import geodesic
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Setup complete.')

---
## Data Loading

Run the cell below to load the training and test datasets. **Do not modify this cell.**

You have two files:
- **`train.csv`**: Features and labels for Stages 1–2. Includes `income_gt50k` (for all), `if_donated_last_year` (observed only for previously contacted individuals, NaN otherwise), and a `previously_contacted` flag.
- **`test.csv`**: Same structure as training, plus `donated` (ground-truth future donation outcome) and location coordinates (`lat`, `lon`). The test set is used both for model evaluation in Stages 1–2 and as the candidate pool for routing in Stage 3.

In [ ]:
# ── Load data ──────────────────────────────────────────────
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Train: {len(train):,} rows  |  Test: {len(test):,} rows')
print(f'\nPreviously contacted rate (train): {train["previously_contacted"].mean():.1%}')
print(f'Donation labels available (train): {train["if_donated_last_year"].notna().sum():,} '
      f'out of {len(train):,} ({train["if_donated_last_year"].notna().mean():.1%})')

# ── Feature definitions ────────────────────────────────────
FEATURE_COLS = ['age', 'workclass', 'education_level', 'education-num', 'marital-status',
                'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
                'hours-per-week', 'native-country']

CAT_COLS = ['workclass', 'education_level', 'marital-status', 'occupation',
            'relationship', 'race', 'sex', 'native-country']

NUM_COLS = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

### Helper Functions

A shared feature-encoding function for consistent preprocessing. **Do not modify this cell.**

In [ ]:
def encode_features(train_data, test_data, cat_cols=CAT_COLS, num_cols=NUM_COLS):
    """
    One-hot encode categorical features, align columns between train and test.
    Returns X_train, X_test as numpy arrays.
    Also stores the dummy column names in encode_features.dummy_cols_ for reuse.
    """
    all_data = pd.concat([train_data, test_data], ignore_index=True)
    all_dummies = pd.get_dummies(all_data[cat_cols], drop_first=True)
    dummy_cols = all_dummies.columns.tolist()
    encode_features.dummy_cols_ = dummy_cols
    encode_features.num_cols_ = num_cols
    encode_features.cat_cols_ = cat_cols

    def _get_X(data):
        X_num = data[num_cols].fillna(0).values
        dummies = pd.get_dummies(data[cat_cols], drop_first=True)
        for c in dummy_cols:
            if c not in dummies.columns:
                dummies[c] = 0
        return np.hstack([X_num, dummies[dummy_cols].values])

    return _get_X(train_data), _get_X(test_data)

def encode_new_data(data):
    """
    Encode new data using the column schema from the last encode_features() call.
    Use this for candidates or any data not seen during training.
    """
    dummy_cols = encode_features.dummy_cols_
    num_cols = encode_features.num_cols_
    cat_cols = encode_features.cat_cols_
    X_num = data[num_cols].fillna(0).values
    dummies = pd.get_dummies(data[cat_cols], drop_first=True)
    for c in dummy_cols:
        if c not in dummies.columns:
            dummies[c] = 0
    return np.hstack([X_num, dummies[dummy_cols].values])

print('Helper functions loaded.')

### How to Use the Helper Functions

**Feature encoding (Stages 1–2):**

```python
# Call encode_features first to fit the encoding on train + test
X_train, X_test = encode_features(train, test)

# Later, to encode new data (e.g., routing candidates in Stage 3),
# use encode_new_data which reuses the same column schema:
X_new = encode_new_data(some_dataframe)
```

You must call `encode_features` before `encode_new_data`. The encoding aligns dummy columns across datasets automatically.

---
## Stage 1: Predicting High-Income Individuals (25 points)

Build and evaluate classification models to predict:

> **`income_gt50k`**: whether an individual's income exceeds $50,000.

This is a **proxy task**. It is not the final business objective, but it lets you practice supervised learning and evaluation. The `income_gt50k` label is available for all individuals in both the training and test sets.

### 1a. Preprocessing (5 points)

Preprocess the features for modeling:
- Handle categorical variables (e.g., one-hot encoding)
- Handle missing values if any
- Report the number of features after preprocessing

💻 **Code required**

In [ ]:
# YOUR CODE HERE

# Encode features using the provided helper
X_train, X_test = encode_features(train, test)

# Report shapes / feature counts
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of features after preprocessing:", X_train.shape[1])

# Check missing values in original feature columns
print("\nMissing values in training features:")
print(train[FEATURE_COLS].isna().sum())

print("\nMissing values in test features:")
print(test[FEATURE_COLS].isna().sum())

### 1b. Train at Least Two Classifiers (10 points)

Train at least two classifiers to predict `income_gt50k`. Examples: logistic regression, random forest, gradient boosting.

For each model, report on the test set:
- Accuracy
- Precision and recall
- AUC

Plot ROC curves for all models on a single figure.

💻 **Code required**

📊 **Report:** comparison table and ROC curve figure

In [ ]:
# YOUR CODE HERE

# Labels
y_train = train['income_gt50k']
y_test = test['income_gt50k']

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    )
}

# Train, evaluate, and store results
results = []
roc_data = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_score)

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_data[name] = (fpr, tpr, auc)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "AUC": auc
    })

# Comparison table
results_df = pd.DataFrame(results).sort_values(by="AUC", ascending=False)
print("Model comparison on test set:")
display(results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "AUC": "{:.4f}"
}))

# ROC curves
plt.figure(figsize=(8, 6))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Income > 50K Classifiers")
plt.legend()
plt.show()

### 1c. Written Questions (10 points)

**Q1 (5 pts):** Which model performed best? Which metric do you consider most informative for this task, and why?

**Q2 (5 pts):** What are the limitations of using income as a proxy for fundraising targeting? Name at least two specific failure modes.

✍️ **Written answer**

*Your answer:*Q1

Gradient Boosting performed the best among the three models. It had the highest accuracy, precision, recall, and AUC on the test set, so overall it was the strongest model for predicting whether income is greater than 50k.

I think AUC is the most informative metric for this task. This is because AUC tells us how well the model separates the positive and negative classes across different thresholds, instead of only looking at one cutoff point. Since the goal is to compare models and understand which one ranks people better overall, AUC is more useful than accuracy alone. Accuracy can sometimes hide important differences, especially if one class is more common than the other.

Q2

Using income as a proxy for fundraising potential has clear limitations because having a high income does not necessarily mean someone will donate. Income may reflect ability to give, but it does not directly measure willingness to give, interest in the cause, or responsiveness to outreach.

One failure mode is that the model may identify people with high income who are actually unlikely to donate. For example, someone may have strong financial capacity but no interest in charity, so targeting them would not help fundraising much.

A second failure mode is that the model may miss lower-income individuals who are still likely to donate. Some people may not earn as much, but they may care a lot about the organization’s mission and be very willing to contribute. In that case, the proxy model would rank them too low and CharityAI could overlook good potential donors.

Another limitation is that income-based targeting could bias outreach toward more advantaged groups, which may not match the nonprofit’s real objective or values. So even if the model predicts income well, it may still not lead to the best fundraising decisions.

---
## Stage 2: Predicting Past Donation (35 points)

Now you use a label closer to the fundraising goal:

> **`if_donated_last_year`**: equals 1 if the individual donated last year, 0 otherwise.

This label is only observed for individuals who were previously contacted (`previously_contacted == 1`). For the remaining ~50% of individuals, the label is missing (NaN). This is a realistic **cold-start** situation.

### 2a. Train a Donation Prediction Model (10 points)

Train one or more classifiers to predict `if_donated_last_year`, using only the training rows where this label is observed (`previously_contacted == 1`). Evaluate on the Known test rows (also `previously_contacted == 1`).

Report accuracy, precision/recall, and AUC.

💻 **Code required**

📊 **Report:** performance metrics table

In [ ]:
# YOUR CODE HERE
# 2a. Train donation prediction model using only observed labels

# Masks for rows where donation label is observed
train_known_mask = (train["previously_contacted"] == 1) & (train["if_donated_last_year"].notna())
test_known_mask  = (test["previously_contacted"] == 1) & (test["if_donated_last_year"].notna())

# Subset encoded features
X_train_known = X_train[train_known_mask]
X_test_known = X_test[test_known_mask]

# Subset labels
y_train_known = train.loc[train_known_mask, "if_donated_last_year"].astype(int)
y_test_known = test.loc[test_known_mask, "if_donated_last_year"].astype(int)

print("Known train rows:", X_train_known.shape[0])
print("Known test rows:", X_test_known.shape[0])
print("Train donation rate:", y_train_known.mean().round(4))
print("Test donation rate:", y_test_known.mean().round(4))

# Define models
donation_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    )
}

# Train and evaluate
donation_results = []

for name, model in donation_models.items():
    model.fit(X_train_known, y_train_known)

    y_pred = model.predict(X_test_known)
    y_score = model.predict_proba(X_test_known)[:, 1]

    donation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_known, y_pred),
        "Precision": precision_score(y_test_known, y_pred),
        "Recall": recall_score(y_test_known, y_pred),
        "AUC": roc_auc_score(y_test_known, y_score)
    })

donation_results_df = (
    pd.DataFrame(donation_results)
    .sort_values(by="AUC", ascending=False)
    .reset_index(drop=True)
)

print("\nDonation model performance on Known test rows:")
print(donation_results_df.round(4))

### 2b. Compare Stage 1 vs Stage 2 for Predicting Future Donations (10 points)

The test set includes a ground-truth column `donated` (the true future donation outcome) for all individuals. Use this to compare how well the Stage 1 model (income proxy) and Stage 2 model (donation label) predict **future donations** on the full test set.

For each model, compute AUC for predicting `donated` on the **entire** test set. Which model appears to perform better overall?

💻 **Code required**

📊 **Report:** AUC comparison table

In [ ]:
# YOUR CODE HERE
# 2b. Compare Stage 1 vs Stage 2 for predicting future donations

# Future donation ground truth on full test set
y_future = test["donated"].astype(int)

# Refit the best Stage 1 model on the full Stage 1 training task
stage1_best_model = GradientBoostingClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE
)
stage1_best_model.fit(X_train, train["income_gt50k"])

# Refit the best Stage 2 model on observed donation labels only
stage2_best_model = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE
)
stage2_best_model.fit(X_train_known, y_train_known)

# Predict scores on the FULL test set
stage1_scores_full = stage1_best_model.predict_proba(X_test)[:, 1]
stage2_scores_full = stage2_best_model.predict_proba(X_test)[:, 1]

# Compute AUC against true future donation outcome
stage1_auc_future = roc_auc_score(y_future, stage1_scores_full)
stage2_auc_future = roc_auc_score(y_future, stage2_scores_full)

auc_comparison_df = pd.DataFrame({
    "Stage": ["Stage 1", "Stage 2"],
    "Training Label": ["income_gt50k", "if_donated_last_year"],
    "Best Model": ["Gradient Boosting", "Logistic Regression"],
    "AUC for Predicting donated": [stage1_auc_future, stage2_auc_future]
})

auc_comparison_df.round(4)

### 2c. Balance Check: Performance by Subgroup (10 points)

The overall AUC comparison in 2b may hide important differences. Split the test set into two subgroups:
- **Known**: individuals with `previously_contacted == 1`
- **Cold**: individuals with `previously_contacted == 0` (the cold-start population)

Compute the AUC of each model (Stage 1 and Stage 2) for predicting `donated` **separately** on each subgroup.

💻 **Code required**

📊 **Report:** 2x2 table (rows = models, columns = subgroups)

✍️ **Written answer** (3-5 sentences): What do you observe? Does the overall winner from 2b win on both subgroups? If not, what explains the reversal? Propose a practical strategy that uses this insight.

In [ ]:
# 2c. Balance check: performance by subgroup

# Subgroup masks
known_mask = test["previously_contacted"] == 1
cold_mask = test["previously_contacted"] == 0

# Use the prediction scores already computed in 2b
stage1_known_auc = roc_auc_score(y_future[known_mask], stage1_scores_full[known_mask])
stage1_cold_auc  = roc_auc_score(y_future[cold_mask], stage1_scores_full[cold_mask])

stage2_known_auc = roc_auc_score(y_future[known_mask], stage2_scores_full[known_mask])
stage2_cold_auc  = roc_auc_score(y_future[cold_mask], stage2_scores_full[cold_mask])

# 2x2 table
subgroup_auc_df = pd.DataFrame({
    "Known": [stage1_known_auc, stage2_known_auc],
    "Cold": [stage1_cold_auc, stage2_cold_auc]
}, index=["Stage 1", "Stage 2"])

print("AUC by subgroup for predicting future donations:")
subgroup_auc_df.round(4)

*Your answer:*The results show that the overall winner from 2b does not win on both subgroups. Stage 2 works better for the Known group, probably because it was trained using donation labels from previously contacted people, so it matches that population better. But on the Cold group, Stage 1 does better, which suggests that the Stage 2 model does not generalize as well to new individuals with no contact history. In practice, I would use Stage 2 for previously contacted people and Stage 1 for cold-start individuals.

### 2d. Written Questions (5 points)

**Q1 (2 pts):** Why is `if_donated_last_year` a more relevant label than `income_gt50k` for fundraising targeting?

**Q2 (3 pts):** Name a specific way in which last year's donation behavior could be misleading for *future* targeting. (Hint: examine the test set. Are there groups that donated last year but are unlikely to donate again, or vice versa?)

✍️ **Written answer**

In [ ]:
# YOUR CODE HERE

*Your answer:*Q1

if_donated_last_year is more relevant than income_gt50k because it is closer to the true business goal of predicting future donations. Income captures ability to give, but past donation captures actual giving behavior, which is more directly related to fundraising targeting.

Q2

Last year’s donation behavior can be misleading because it does not perfectly carry over into the future. In the Known test group, about 37.3% of people who donated last year do not donate this year, while about 50.1% of people who did not donate last year end up donating this year. For example, among previously contacted people age 30 or younger who donated last year, only about 45.4% donate again, but among people age 60+ who did not donate last year, about 64.0% donate this year. This shows that past donation is useful, but it should not be treated as a complete rule for future targeting.

---
## Stage 3: Spatial Targeting and Routing Under Constraints (40 points)

Now you move from prediction to decision-making. CharityAI sends field representatives to visit potential donors **in person** in the Pudong area of Shanghai. Each visit generates a donation with some probability, but travel between candidates costs time and money. Your task is to decide **whom to visit** and **in what order**, given a limited travel budget.

**Model selection:** Based on Stages 1–2, choose the model (or hybrid strategy) you believe is best suited for targeting. Treat it as fixed before proceeding.

### Setting

- You will use the **test set** as your candidate pool. Each test individual has a location (`lat`, `lon`) and a ground-truth `donated` outcome.
- A field representative starts from a fixed **depot** (NYU Shanghai campus) and must return after completing visits.
- Each visit incurs a **fixed visit cost** of ¥5 plus a **travel cost** of ¥10 per kilometer from the previous stop.
- Each visited individual either donates (¥200) or does not (¥0). You use your model's predicted probability to *plan* the route, but **realized profit** is computed from the actual `donated` outcome.
- **Maximum travel distance**: 15 km total (including return to depot).

### Objective

Plan a route to maximize **realized net profit**:

$$\text{Realized Net Profit} = \sum_{i \in \text{visited}} d_i \times 200 - \text{Total Cost}$$

where $d_i \in \{0, 1\}$ is the actual donation outcome, and Total Cost = (number of visits × ¥5) + (total distance traveled × ¥10/km).

You design the route using predicted probabilities $p_i$, but the final evaluation uses the true outcomes.

### Data and Visualization Setup

Run the setup and visualization cells below. **Do not modify the parameter cell (Stage 3 parameters) or the visualization helpers cell.** You will need to fill in the prediction cell to assign `routing_df['p_donate']`.

In [ ]:
# ── Stage 3 parameters ────────────────────────────────────
DEPOT = {'lat': 31.1508, 'lon': 121.4770}  # NYU Shanghai, 567 West Yangsi Road
DONATION_VALUE = 200   # ¥ per donation
VISIT_COST = 5         # ¥ per visit
TRAVEL_COST_PER_KM = 10  # ¥ per km
MAX_DISTANCE = 15.0    # km total route limit

# ── Subsample from test set for routing (manageable size) ──
np.random.seed(RANDOM_STATE)
N_ROUTING = 200  # number of candidates for the routing problem
routing_idx = np.random.choice(len(test), size=N_ROUTING, replace=False)
routing_df = test.iloc[routing_idx].reset_index(drop=True).copy()

print(f'Routing candidates: {len(routing_df)}')
print(f'Depot: ({DEPOT["lat"]:.4f}, {DEPOT["lon"]:.4f})')
print(f'Max distance: {MAX_DISTANCE} km')
print(f'Donation rate in routing set: {routing_df["donated"].mean():.1%}')

In [ ]:
# ── Generate donation probability predictions for routing candidates ──
# Use the hybrid strategy from 2c:
# Stage 2 for previously contacted individuals, Stage 1 for cold-start individuals

X_routing = encode_new_data(routing_df)

# Predicted probabilities from the two selected models
p_s1 = stage1_best_model.predict_proba(X_routing)[:, 1]   # Stage 1 model
p_s2 = stage2_best_model.predict_proba(X_routing)[:, 1]   # Stage 2 model

# Hybrid assignment
routing_df['p_donate'] = np.where(
    routing_df['previously_contacted'] == 1,
    p_s2,
    p_s1
)

# ── After assigning p_donate, run these lines to verify ──
routing_df['expected_value'] = routing_df['p_donate'] * DONATION_VALUE
print(f'Donation probability stats:')
print(routing_df['p_donate'].describe().round(3))

In [ ]:
# ── Visualization helpers (provided) ──────────────────────

def compute_distance_matrix(locations, depot):
    """Compute pairwise geodesic distance matrix (km) including depot at index 0."""
    all_points = [depot] + [{'lat': r['lat'], 'lon': r['lon']} for _, r in locations.iterrows()]
    n = len(all_points)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = geodesic((all_points[i]['lat'], all_points[i]['lon']),
                         (all_points[j]['lat'], all_points[j]['lon'])).kilometers
            matrix[i][j] = matrix[j][i] = d
    return matrix, all_points

def evaluate_route(route, dist_matrix, donation_probs, actual_donated=None,
                   donation_value=DONATION_VALUE,
                   visit_cost=VISIT_COST, travel_cost_per_km=TRAVEL_COST_PER_KM):
    """Evaluate a route. Route is a list of indices (0=depot, 1..N=candidates).
    Reports both expected profit (from model predictions) and realized profit
    (from actual donation outcomes, if provided)."""
    total_dist = sum(dist_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
    visited = [i for i in route if i != 0]  # exclude depot
    n_visited = len(visited)
    total_cost = n_visited * visit_cost + total_dist * travel_cost_per_km
    expected_revenue = sum(donation_probs[i-1] * donation_value for i in visited)
    expected_profit = expected_revenue - total_cost
    result = {
        'route': route, 'n_visited': n_visited, 'total_distance_km': total_dist,
        'total_cost': total_cost, 'expected_revenue': expected_revenue,
        'expected_profit': expected_profit,
    }
    if actual_donated is not None:
        realized_revenue = sum(actual_donated[i-1] * donation_value for i in visited)
        n_donors = sum(actual_donated[i-1] for i in visited)
        result['realized_revenue'] = realized_revenue
        result['realized_profit'] = realized_revenue - total_cost
        result['n_donors'] = int(n_donors)
        result['hit_rate'] = n_donors / n_visited if n_visited > 0 else 0
    return result

def plot_route_map(all_points, route, donation_probs, title='Route'):
    """Create a folium map showing a route with color-coded candidates."""
    center = [DEPOT['lat'], DEPOT['lon']]
    m = folium.Map(location=center, zoom_start=15, tiles='CartoDB positron')

    # Depot
    folium.Marker(center, icon=folium.Icon(color='black', icon='home'),
                  popup='Depot').add_to(m)

    # Route line
    route_coords = [[all_points[i]['lat'], all_points[i]['lon']] for i in route]
    folium.PolyLine(route_coords, color='#2E8B57', weight=3, opacity=0.8).add_to(m)

    # Candidates
    visited_set = set(route)
    for idx in range(1, len(all_points)):
        p = donation_probs[idx - 1]
        color = 'green' if idx in visited_set else 'gray'
        radius = 4 + p * 12
        folium.CircleMarker(
            [all_points[idx]['lat'], all_points[idx]['lon']],
            radius=radius, color=color, fill=True, fill_color=color, fill_opacity=0.7,
            popup=f'ID: {idx}<br>P(donate): {p:.2f}'
        ).add_to(m)

    return m

def plot_scatter_selection(all_points, route, donation_probs, depot, title=''):
    """Scatter plot: distance from depot vs donation probability, colored by selection."""
    visited_set = set(route)
    dists = [geodesic((all_points[i]['lat'], all_points[i]['lon']),
                      (depot['lat'], depot['lon'])).kilometers
             for i in range(1, len(all_points))]

    colors = ['#2E8B57' if (i+1) in visited_set else '#CCCCCC' for i in range(len(dists))]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(dists, donation_probs, c=colors, s=60, alpha=0.8, edgecolors='white', linewidth=0.5)
    ax.set_xlabel('Distance from Depot (km)')
    ax.set_ylabel('P(donate)')
    ax.set_title(title)

    # Legend
    ax.scatter([], [], c='#2E8B57', s=60, label='Visited')
    ax.scatter([], [], c='#CCCCCC', s=60, label='Not visited')
    ax.legend()
    plt.tight_layout()
    return fig

# Precompute distance matrix
dist_matrix, all_points = compute_distance_matrix(routing_df, DEPOT)
donation_probs = routing_df['p_donate'].values
actual_donated = routing_df['donated'].values  # ground truth for realized profit

print(f'Distance matrix computed: {dist_matrix.shape}')
print(f'Max pairwise distance: {dist_matrix.max():.2f} km')
print(f'Ground truth donation rate: {actual_donated.mean():.1%} ({actual_donated.sum()} donors out of {len(actual_donated)})')

### How to Use the Stage 3 Helper Functions

**Route format:** A route is a list of integers starting and ending with `0` (the depot). Candidate indices run from `1` to `N` (matching the distance matrix). For example, `[0, 5, 12, 3, 0]` means: depart depot → visit candidate 5 → visit candidate 12 → visit candidate 3 → return to depot.

**Distance matrix:** `dist_matrix` is an `(N+1) × (N+1)` matrix where index 0 is the depot and indices 1..N are candidates. `dist_matrix[i][j]` gives the distance in km between locations `i` and `j`.

**Evaluating a route:**

```python
result = evaluate_route(route, dist_matrix, donation_probs, actual_donated)
# result is a dictionary with keys:
#   'n_visited'          - number of candidates visited
#   'total_distance_km'  - total route distance
#   'total_cost'         - visit costs + travel costs
#   'expected_profit'    - profit using predicted probabilities
#   'realized_profit'    - profit using actual donation outcomes
#   'n_donors'           - number of visited candidates who actually donated
#   'hit_rate'           - fraction of visited candidates who donated
```

**Checking feasibility:** Before adding a candidate to your route, verify that the total distance (including return to depot) stays within `MAX_DISTANCE`:

```python
d_to_candidate = dist_matrix[current_position][candidate]
d_back_to_depot = dist_matrix[candidate][0]
if total_dist_so_far + d_to_candidate + d_back_to_depot <= MAX_DISTANCE:
    # feasible: add candidate to route
```

**Visualization:**

```python
# Scatter plot: distance from depot vs P(donate), colored by selection
fig = plot_scatter_selection(all_points, route, donation_probs, DEPOT,
                             title='My Strategy')
plt.show()

# Interactive map with route drawn on it
m = plot_route_map(all_points, route, donation_probs, title='My Route')
m  # display in notebook
```

**Key variables available** (from the setup cells above):
- `routing_df` : DataFrame of 200 candidates with features, `p_donate`, and `donated`
- `dist_matrix` : pairwise distance matrix (201 × 201, index 0 = depot)
- `all_points` : list of dicts with `lat`/`lon` (index 0 = depot, 1..200 = candidates)
- `donation_probs` : numpy array of predicted donation probabilities (length 200)
- `actual_donated` : numpy array of ground-truth donation outcomes (length 200)
- `DONATION_VALUE`, `VISIT_COST`, `TRAVEL_COST_PER_KM`, `MAX_DISTANCE` : cost parameters

### 3a. Two Baseline Strategies (10 points)

Implement and evaluate the following two baseline routing strategies:

**Baseline 1: Top Scores First (value-driven):** Rank all candidates by $p_i$ (donation probability) in descending order. Visit them in that order until the distance budget is exhausted. This strategy maximizes predicted value but ignores geography.

**Baseline 2: Nearest First (distance-driven):** Starting from the depot, always visit the nearest unvisited candidate next, until the distance budget is exhausted. This strategy minimizes travel waste but ignores donation probability.

For each baseline, report: number of candidates visited, realized profit, and hit rate. Show a scatter plot (distance from depot vs. $p_i$) with visited candidates highlighted.

💻 **Code required**

📊 **Report:** comparison table, two scatter plots

✍️ **Written answer** (3-5 sentences): Compare the two baselines. Which visits more candidates? Which achieves higher realized profit? What does each strategy sacrifice?

In [ ]:
# 3a. Two baseline strategies

# ---------- Helper functions ----------
def build_route_from_order(order, dist_matrix, max_distance=MAX_DISTANCE):
    """
    Build a feasible route by following a proposed candidate order.
    Route always starts and ends at depot (0).
    """
    route = [0]
    current = 0
    total_dist = 0.0
    visited = set()

    for candidate in order:
        if candidate in visited:
            continue

        d_to_candidate = dist_matrix[current][candidate]
        d_back_to_depot = dist_matrix[candidate][0]

        # Must still be able to return to depot
        if total_dist + d_to_candidate + d_back_to_depot <= max_distance:
            route.append(candidate)
            visited.add(candidate)
            total_dist += d_to_candidate
            current = candidate

    route.append(0)
    return route


def nearest_feasible_route(dist_matrix, max_distance=MAX_DISTANCE):
    """
    Greedy nearest-first baseline:
    from current position, always choose the nearest feasible unvisited candidate.
    """
    n_candidates = dist_matrix.shape[0] - 1
    unvisited = set(range(1, n_candidates + 1))

    route = [0]
    current = 0
    total_dist = 0.0

    while True:
        feasible = []
        for candidate in unvisited:
            d_to_candidate = dist_matrix[current][candidate]
            d_back_to_depot = dist_matrix[candidate][0]

            if total_dist + d_to_candidate + d_back_to_depot <= max_distance:
                feasible.append(candidate)

        if not feasible:
            break

        next_candidate = min(feasible, key=lambda c: dist_matrix[current][c])
        total_dist += dist_matrix[current][next_candidate]
        route.append(next_candidate)
        unvisited.remove(next_candidate)
        current = next_candidate

    route.append(0)
    return route


# ---------- Baseline 1: Top Scores First ----------
score_order = np.argsort(-donation_probs) + 1   # candidate indices are 1..N
route_top_scores = build_route_from_order(score_order, dist_matrix)

result_top_scores = evaluate_route(
    route_top_scores, dist_matrix, donation_probs, actual_donated
)

# ---------- Baseline 2: Nearest First ----------
route_nearest = nearest_feasible_route(dist_matrix)

result_nearest = evaluate_route(
    route_nearest, dist_matrix, donation_probs, actual_donated
)

# ---------- Comparison table ----------
baseline_results_df = pd.DataFrame([
    {
        "Strategy": "Top Scores First",
        "Candidates Visited": result_top_scores["n_visited"],
        "Realized Profit": result_top_scores["realized_profit"],
        "Hit Rate": result_top_scores["hit_rate"]
    },
    {
        "Strategy": "Nearest First",
        "Candidates Visited": result_nearest["n_visited"],
        "Realized Profit": result_nearest["realized_profit"],
        "Hit Rate": result_nearest["hit_rate"]
    }
])

print("Baseline strategy comparison:")
display(baseline_results_df.round(4))

# ---------- Scatter plot: Baseline 1 ----------
fig1 = plot_scatter_selection(
    all_points,
    route_top_scores,
    donation_probs,
    DEPOT,
    title="Baseline 1: Top Scores First"
)
plt.show()

# ---------- Scatter plot: Baseline 2 ----------
fig2 = plot_scatter_selection(
    all_points,
    route_nearest,
    donation_probs,
    DEPOT,
    title="Baseline 2: Nearest First"
)
plt.show()

*Your answer:*The results show a clear difference between the two baselines. Top Scores First has a much higher hit rate because it chooses people with higher predicted donation probability, but it only visits 19 candidates and ends up with much lower realized profit. Nearest First visits many more people because it uses the travel budget more efficiently, and this gives it a much higher realized profit overall. So Top Scores First sacrifices geography, while Nearest First sacrifices donation quality. This suggests that a better strategy should balance both probability and distance.

### 3b. Propose a Combined Strategy (10 points)

The two baselines in 3a represent two extremes: one ignores geography, the other ignores donation probability. Design a heuristic that accounts for **both** predicted donation probability and geographic location. Possible approaches include:
- Score candidates by expected donation per unit travel cost ("bang for the buck")
- Cluster candidates geographically, then select and route within the best cluster
- Prioritize dense areas where many promising candidates are close together
- Filter by a probability threshold, then route by nearest neighbor within the filtered set

**Describe your approach clearly before implementing it.** You do not need a mathematically optimal solution; a clear, well-justified heuristic is acceptable.

Compare your strategy to both baselines from 3a.

**Performance requirement:** Your combined strategy must achieve a realized profit of at least **¥9,465** to receive full credit. This threshold is achievable with a reasonable heuristic.

💻 **Code required**

📊 **Report:** route evaluation metrics (three-way comparison table), scatter plot, map

✍️ **Written answer** (describe your heuristic in 3-5 sentences before the code)

*Your answer:*My combined strategy uses both predicted donation probability and geographic efficiency. First, I filter out candidates with very low predicted donation probability, since those visits are unlikely to justify the visit and travel costs. Then, among the remaining candidates, I construct the route using a nearest-neighbor rule so that the travel budget is spent efficiently. To avoid choosing the threshold based on realized outcomes, I evaluate a small grid of probability thresholds using expected profit only and select the threshold that gives the best expected route. This balances donor quality and routing efficiency better than either baseline alone.

In [ ]:
# YOUR CODE HERE
# 3b. Combined strategy: probability threshold + nearest-neighbor routing

def nearest_route_on_subset(candidate_indices, dist_matrix, max_distance=MAX_DISTANCE):
    """
    Build a feasible route using nearest-neighbor, but only over a specified subset
    of candidate indices (1..N).
    """
    unvisited = set(candidate_indices)
    route = [0]
    current = 0
    total_dist = 0.0

    while True:
        feasible = []
        for candidate in unvisited:
            d_to_candidate = dist_matrix[current][candidate]
            d_back_to_depot = dist_matrix[candidate][0]

            if total_dist + d_to_candidate + d_back_to_depot <= max_distance:
                feasible.append(candidate)

        if not feasible:
            break

        next_candidate = min(feasible, key=lambda c: dist_matrix[current][c])
        total_dist += dist_matrix[current][next_candidate]
        route.append(next_candidate)
        unvisited.remove(next_candidate)
        current = next_candidate

    route.append(0)
    return route


# ---------------------------------------------------
# Search over probability thresholds using EXPECTED profit only
# ---------------------------------------------------
threshold_grid = np.arange(0.05, 0.55, 0.05)

search_results = []
best_threshold = None
best_route = None
best_expected_profit = -np.inf

for threshold in threshold_grid:
    candidate_indices = np.where(donation_probs >= threshold)[0] + 1  # convert to 1..N indexing

    if len(candidate_indices) == 0:
        continue

    route = nearest_route_on_subset(candidate_indices, dist_matrix, max_distance=MAX_DISTANCE)
    expected_result = evaluate_route(route, dist_matrix, donation_probs)  # no actual outcomes used here

    search_results.append({
        "Threshold": threshold,
        "Candidates Visited": expected_result["n_visited"],
        "Expected Profit": expected_result["expected_profit"],
        "Total Distance (km)": expected_result["total_distance_km"]
    })

    if expected_result["expected_profit"] > best_expected_profit:
        best_expected_profit = expected_result["expected_profit"]
        best_threshold = threshold
        best_route = route

# Final evaluation using actual donation outcomes
result_combined = evaluate_route(best_route, dist_matrix, donation_probs, actual_donated)

print(f"Chosen probability threshold: {best_threshold:.2f}")
print(f"Best expected profit (used for selection): {best_expected_profit:.2f}")

# Optional: show threshold search summary
threshold_search_df = pd.DataFrame(search_results).sort_values(by="Expected Profit", ascending=False)
print("\nThreshold search (sorted by expected profit):")
display(threshold_search_df.round(4))

# ---------------------------------------------------
# Three-way comparison table
# Assumes result_top_scores and result_nearest already exist from 3a
# ---------------------------------------------------
comparison_3way_df = pd.DataFrame([
    {
        "Strategy": "Top Scores First",
        "Threshold": np.nan,
        "Candidates Visited": result_top_scores["n_visited"],
        "Total Distance (km)": result_top_scores["total_distance_km"],
        "Expected Profit": result_top_scores["expected_profit"],
        "Realized Profit": result_top_scores["realized_profit"],
        "Hit Rate": result_top_scores["hit_rate"]
    },
    {
        "Strategy": "Nearest First",
        "Threshold": np.nan,
        "Candidates Visited": result_nearest["n_visited"],
        "Total Distance (km)": result_nearest["total_distance_km"],
        "Expected Profit": result_nearest["expected_profit"],
        "Realized Profit": result_nearest["realized_profit"],
        "Hit Rate": result_nearest["hit_rate"]
    },
    {
        "Strategy": "Combined: Threshold + Nearest",
        "Threshold": best_threshold,
        "Candidates Visited": result_combined["n_visited"],
        "Total Distance (km)": result_combined["total_distance_km"],
        "Expected Profit": result_combined["expected_profit"],
        "Realized Profit": result_combined["realized_profit"],
        "Hit Rate": result_combined["hit_rate"]
    }
])

print("\nThree-way comparison:")
display(comparison_3way_df.round(4))

# ---------------------------------------------------
# Scatter plot for combined strategy
# ---------------------------------------------------
fig = plot_scatter_selection(
    all_points,
    best_route,
    donation_probs,
    DEPOT,
    title=f"Combined Strategy (threshold = {best_threshold:.2f})"
)
plt.show()

# ---------------------------------------------------
# Interactive map for combined strategy
# ---------------------------------------------------
combined_map = plot_route_map(
    all_points,
    best_route,
    donation_probs,
    title=f"Combined Route (threshold = {best_threshold:.2f})"
)
combined_map


In [ ]:
# YOUR CODE HERE

### 3c. Budget Sensitivity Analysis (10 points)

Management asks: *"If we increase the maximum travel distance, will we actually visit more people and raise more money?"*

Evaluate your combined strategy from 3b under multiple distance budgets (e.g., 2, 5, 8, 10, 15, 20, 30 km) and answer:

1. Does increasing the distance budget always increase **realized** profit?
2. Where do you see diminishing returns?
3. As the budget grows, what types of candidates get added at the margin (high probability but far away, or moderate probability but nearby)?

Present your findings as a table and/or plot. Report both expected profit (from model predictions) and realized profit (from actual outcomes).

💻 **Code required**

📊 **Report:** table or plot showing profit vs. budget

✍️ **Written answer** (3-5 sentences)

In [ ]:
# YOUR CODE HERE
# 3c. Budget sensitivity analysis for the combined strategy

# Use the same combined strategy from 3b:
# fixed threshold chosen in 3b + nearest-neighbor routing within that subset

budget_list = [2, 5, 8, 10, 15, 20, 30]

# Candidate pool selected by the 3b threshold
selected_candidates = np.where(donation_probs >= best_threshold)[0] + 1  # 1..N indexing

budget_results = []
prev_visited_set = set()

for budget in budget_list:
    # Rebuild route under this distance budget
    route_b = nearest_route_on_subset(selected_candidates, dist_matrix, max_distance=budget)
    result_b = evaluate_route(route_b, dist_matrix, donation_probs, actual_donated)

    visited_set = set([i for i in route_b if i != 0])
    newly_added = sorted(list(visited_set - prev_visited_set))

    # Summarize the marginal additions when budget increases
    if len(newly_added) > 0:
        avg_p_added = np.mean([donation_probs[i - 1] for i in newly_added])
        avg_dist_added = np.mean([dist_matrix[0][i] for i in newly_added])  # distance from depot
    else:
        avg_p_added = np.nan
        avg_dist_added = np.nan

    budget_results.append({
        "Budget (km)": budget,
        "Candidates Visited": result_b["n_visited"],
        "Total Distance (km)": result_b["total_distance_km"],
        "Expected Profit": result_b["expected_profit"],
        "Realized Profit": result_b["realized_profit"],
        "Hit Rate": result_b["hit_rate"],
        "Newly Added Candidates": len(newly_added),
        "Avg P(donate) of New Additions": avg_p_added,
        "Avg Distance from Depot of New Additions": avg_dist_added
    })

    prev_visited_set = visited_set

budget_df = pd.DataFrame(budget_results)

print("Budget sensitivity table:")
display(budget_df.round(4))

# --- Plot 1: Expected vs realized profit ---
plt.figure(figsize=(8, 5))
plt.plot(budget_df["Budget (km)"], budget_df["Expected Profit"], marker='o', label="Expected Profit")
plt.plot(budget_df["Budget (km)"], budget_df["Realized Profit"], marker='o', label="Realized Profit")
plt.xlabel("Maximum Travel Distance (km)")
plt.ylabel("Profit")
plt.title("Combined Strategy: Profit vs Distance Budget")
plt.legend()
plt.show()

# --- Plot 2: Number of visited candidates ---
plt.figure(figsize=(8, 5))
plt.plot(budget_df["Budget (km)"], budget_df["Candidates Visited"], marker='o')
plt.xlabel("Maximum Travel Distance (km)")
plt.ylabel("Candidates Visited")
plt.title("Combined Strategy: Visits vs Distance Budget")
plt.show()

*Your answer:*In my results, increasing the travel budget does increase realized profit, but not by the same amount each time. Profit grows a lot from 2 km to 15 km, but after that the increase becomes much smaller, so there are clear diminishing returns. For example, going from 20 km to 30 km only adds 2 more visited candidates and increases realized profit by less than ¥200. At smaller budgets, the added candidates are mostly reasonably strong and still fairly close, but at larger budgets the route starts adding farther and weaker marginal candidates. This means that giving the team more travel distance helps at first, but after around 15–20 km the extra value becomes limited.

### 3d. Written Questions (10 points)

**Q1 (4 pts):** In your strategy, how did you balance a candidate's predicted donation probability against their geographic accessibility? What is the core trade-off?

**Q2 (3 pts):** In 3-5 sentences, describe your approach to a manager who cares about the reasoning, not the technical details.

**Q3 (3 pts):** Think of one realistic way the environment or constraints could change (e.g., multiple field representatives, time windows, variable donation amounts). Which part of your current strategy would be most affected, and why?

✍️ **Written answer**

*Your answer:*Q1

In my strategy, I balanced donation probability and geographic accessibility in two steps. First, I filtered out candidates with very low predicted donation probability, so the route would focus on people who were more likely to generate value. Second, within the remaining set, I used a nearest-neighbor routing rule so that travel distance was used efficiently. The core trade-off is that high-probability donors are not always close by, while nearby candidates are not always strong donation prospects. A good route therefore has to balance donor quality against travel efficiency, rather than optimizing only one of them.

Q2

My approach was to avoid the two extreme strategies. Instead of visiting only the highest-scoring people regardless of distance, or visiting only the closest people regardless of donation likelihood, I first removed the weakest candidates and then built a route among the stronger ones in a geographically efficient way. This gave the team a route that still targeted promising donors but did not waste too much travel budget. In practice, the strategy worked better because it combined good targeting with realistic field efficiency.

Q3

One realistic change would be allowing multiple field representatives instead of just one. The part of my strategy that would be most affected is the routing step, because the problem would no longer be about finding one efficient path from a single depot. Instead, we would need to divide candidates across several routes and coordinate who visits whom so that representatives do not overlap and the total coverage is efficient. The prediction model itself could still be useful, but the route-construction heuristic would need to be redesigned for multi-route allocation.

---
## Grading Rubric

| Part | Points | Key Criteria |
|------|--------|-------------|
| **1. Predicting High-Income Individuals** | **25** | |
| 1a. Preprocessing | 5 | Correct encoding, missing value handling, feature count |
| 1b. Train classifiers | 10 | At least two models, correct metrics (accuracy, precision, recall, AUC), ROC curves |
| 1c. Written questions | 10 | Informative metric choice with reasoning; at least two specific proxy limitations |
| **2. Predicting Past Donation** | **35** | |
| 2a. Donation model | 10 | Correct training on Known subset only, performance metrics |
| 2b. Overall comparison | 10 | Correct AUC comparison on full test set using `donated` ground truth |
| 2c. Balance check | 10 | Correct subgroup analysis (Known vs. Cold), identification of reversal, hybrid proposal |
| 2d. Written questions | 5 | Relevant label reasoning; specific failure mode with evidence |
| **3. Spatial Targeting and Routing** | **40** | |
| 3a. Two baselines | 10 | Working top-scores-first and nearest-first implementations, comparison table, clear explanation of trade-offs |
| 3b. Combined strategy | 10 | Clear description before code, working heuristic, three-way comparison, realized profit ≥ ¥9,465 |
| 3c. Budget sensitivity | 10 | At least 4 budget levels, diminishing returns identified, both expected and realized profit reported |
| 3d. Written questions | 10 | Trade-off reasoning (4), manager explanation (3), realistic extension (3) |
| **Total** | **100** | |

---
## Submission Checklist

Before submitting, verify:

- [ ] All code cells run **top-to-bottom** without errors (Kernel → Restart & Run All)
- [ ] All tables, plots, and figures are visible in the notebook output
- [ ] All written-answer cells are filled in
- [ ] You have not modified the data loading, helper function, or parameter cells
- [ ] Stage 1b reports accuracy, precision, recall, and AUC for at least two classifiers
- [ ] Stage 2a trains on the Known subset only (rows where `previously_contacted == 1`)
- [ ] Stage 2c includes the balance check with Known/Cold subgroup AUC comparison
- [ ] Stage 3a implements both top-scores-first and nearest-first baselines
- [ ] Stage 3b describes the heuristic approach *before* the implementation code
- [ ] Stage 3b achieves realized profit ≥ ¥9,465
- [ ] Stage 3 reports both expected and realized profit for all route evaluations
- [ ] Stage 3c includes a budget sensitivity table/plot with at least 4 budget levels